# `ptof_obs_liveness_detection`

## In plain terms
This notebook watches for the pipeline going quiet or slow in ways nobody would notice on
their own — because to everyone downstream, "no new output" and "an output that's just late"
look the same as "nothing happened yet." Seven checks, all CRITICAL (page a human):
1. **A capability stopped producing output** for longer than it plausibly ever has before
   (e.g. no new `saa-display` output for 2+ hours, when historically the longest gap ever seen
   was ~16 minutes).
2. **A capability with irregular timing has gone dark for an unusually long time** (currently
   only `sev2-insights`, which doesn't run on a fixed schedule, so it needs a much longer
   backstop — 120 hours — before flagging).
3. **Output records are missing basic context** (which shift/batch they belong to), which
   silently breaks every report that tries to line up AI output against ISH records.
4. **An upstream data-refresh job failed outright.** The AI is still generating output, but on
   stale source data — nothing else would catch this.
5. **One specific data table stopped refreshing**, even though the rest of the fleet is still
   running on schedule (a real incident: 14 of 19 tables stalled for ~25 hours over the
   2026-09-19/20 weekend while the other 5 kept things looking normal).
6. **A data-refresh job is completing, but much slower than its own history** — an early
   warning that often precedes an outright failure or staleness incident.
7. **A data-refresh job reported success but wrote an abnormal number of rows** — e.g. a table
   that has written a nonzero row count on every run for months suddenly writes zero, while
   still reporting `status='success'`. A silent, undetected partial load.

**If any of these fire:** treat it like item 4 above (stale source data) unless the payload
says otherwise — the AI is probably still "working," just on data or a schedule it shouldn't
be. Check `v_etl_bronze`/`v_llm_bronze` directly (see `BACKTRACK` in `ptof_obs_alert.ipynb`) for
the underlying rows before assuming it's a false alarm.

## What this notebook does (technical detail)
Detects liveness and data-quality gaps across the pipeline:
1. **Capability silence** (CRITICAL, added as WARN; promoted 2026-09-21, evidence-based) — a
   registered capability hasn't produced output in >grace_hours. Promoted on real historical
   evidence: full call-gap distribution for the 3 covered capabilities shows 0 empirical false
   positives ever, with real margin over the grace threshold (saa-display/situational-awareness
   7.4x, summary 3.0x).
2. **Capability silence ceiling** (CRITICAL, added 2026-09-21 as WARN; promoted CRITICAL and
   tightened 168h→120h 2026-09-21) — backstop for capabilities excluded from #1 because
   `silence_grace_hours IS NULL` (irregular cadence); fires only past a much longer fixed
   ceiling (`silence_ceiling_hours`)
3. **Shift context missing** (CRITICAL, added 2026-09-18 as WARN; promoted 2026-09-21) — blank
   shift_date/shift_type/batch_nbr in output records
4. **ETL pipeline health** (CRITICAL) — an upstream ETL task failed, meaning the agent is
   running on stale source data even though it's still producing outputs
5. **ETL table staleness** (CRITICAL, added 2026-09-21) — per-table companion to
   `etl_pipeline_staleness` (the global scalar check in `ptof_obs_alert.ipynb`): flags any single
   ETL source table that hasn't completed a run in >60 min, independent of whether the rest of
   the fleet is still running. Closes a confirmed masking gap — during the 2026-09-19/20
   weekend outage, 14 of 19 tables stalled for ~25h while the other 5 kept the global
   `max(run_timestamp)` fresh, so the global check never fired. Runs alongside the global check,
   not instead of it — see HANDOFF.md for the supplement-vs-replace reasoning (signed off
   2026-09-21).
6. **ETL run slow** (CRITICAL, added 2026-09-18 as WARN; promoted 2026-09-21) — an ETL task's
   duration_seconds has degraded beyond its own MAD-based historical bound, without outright
   failing; rolled up to `(task_name, window_start)` (grain changed 2026-09-21 — see below)
7. **ETL row count anomaly** (CRITICAL, added 2026-09-21, bronze-projection gap review) — an
   ETL run reported `status='success'` on schedule but `rows_written` broke that table's
   all-time-consistent zero/nonzero regime (a silent partial/no-op load). Closes a blind spot
   the other 3 ETL detectors above all share: none of them look at row counts, only
   error/recency/duration. See below for the bimodal-regime evidence.

## Retired detector (2026-09-21 detector value audit)
- **Write lag anomalies** — `write_lag_s` (ingestion_ts - called_at) has been exactly 0 for all
  10,721 rows over 30 days: zero variance, so the MAD-based `greatest(bound, 300s)` guard was
  degenerate and untunable regardless of floor. Not a threshold problem — the signal carried no
  information in prod, whether that reflects a genuinely-instant write path or unwired upstream
  instrumentation. Table and check() removed entirely rather than left WARN-forever with
  nothing to promote. A fresh detector can be added later if write-latency instrumentation is
  ever wired up for real.

## Position in the pipeline
- **Job:** `obs_fresh_scan`, task `02_latency_detection` — runs after `01_bronze_projections`,
  before `06_alert`. (Item 6 is the only genuinely latency-related logic left in this task,
  now that write_lag_anomalies is retired — the task name still doesn't quite match its
  contents.)
- **Upstream:** reads `v_llm_bronze`, `v_etl_bronze` (built by `ptof_obs_bronze_projection`),
  `capability_registry` (human-curated by `ptof_obs_setup_seed`), and (added 2026-09-18)
  `etl_duration_baseline` (built by `ptof_obs_nightly_baseline`).
- **Downstream:** `ptof_obs_alert.ipynb` reads all 7 detectors above (all CRITICAL) directly
  from tables via `INCIDENT_SOURCES`.

## Tables/views touched
- **Reads:** `v_llm_bronze`, `v_etl_bronze`, `capability_registry`, `etl_duration_baseline`
- **Writes:** `capability_silence`, `capability_silence_ceiling`, `shift_context_missing`,
  `etl_pipeline_health`, `etl_table_staleness`, `etl_run_slow`, `etl_row_count_anomaly`

## Capability silence (promoted WARN → CRITICAL 2026-09-21, detector value audit)
Full historical call-gap distribution queried per capability: saa-display and
situational-awareness (4,815 gaps each) never exceeded 0.27h against a 2h grace — a 7.4x
margin; summary (96 gaps) never exceeded 12.04h against a 36h grace — a 3.0x margin. Zero
empirical false positives across the entire history for any of the 3 covered capabilities.
WARN-forever was hiding the exact single-capability-outage failure mode this detector exists
to catch, against a threshold that has real margin rather than slack from being loose.
`finding_signature` added alongside the promotion: a constant `sha2(capability, 256)`, same
lifecycle pattern as `capability_silence_ceiling`/`shift_context_missing`.

## Capability silence ceiling (added 2026-09-21 as WARN item #2; promoted CRITICAL 2026-09-21)
Backstop for any capability with `silence_grace_hours IS NULL` in `capability_registry`
(currently just `sev2-insights`) — those capabilities are structurally excluded from
`capability_silence` above, so before this addition there was no detection path at any length
if one went permanently dark. New `silence_ceiling_hours` column on `capability_registry`
(NULL for every capability except `sev2-insights`).

**Promoted WARN → CRITICAL and tightened 168h → 120h (2026-09-21, FP/FN bias review follow-up,
signed off: "ok lets do it").** Queried against `sev2-insights`'s full history (989 gaps): worst
gap ever observed 102.5h, p99 12.2h, zero gaps ever exceeded 120h or 168h. 120h was chosen over
the original 168h because it still clears the worst historical gap with room (~17%/17.5h margin,
same 0/989 empirical false-positive rate as 168h) while cutting the false-negative exposure
window by 29% — tightening to the evidence floor rather than leaving slack unused. Caveat
carried into `threshold_basis`: this bounds only the *known* failure shape (a 120h+ dark
period) on a single capability's thin history (~572 rows) — it doesn't guarantee no false
negative below that ceiling, doesn't account for future cadence drift, and (like `etl_run_slow`
before its own promotion) hasn't yet fired on a real event, so it should be revisited once it
has some live-fire history.

## Shift context missing (promoted WARN → CRITICAL 2026-09-21, FP/FN bias review step 3)
A standing data-quality gap — blank/null shift_type, batch_nbr, or shift_date on output
records — silently degrades every downstream AI-to-ISH correlation join without erroring, which
was judged not worth leaving invisible to a human indefinitely. Same promote-on-principle basis
as `capability_silence_ceiling` above, not a fresh empirical false-positive re-validation.
`finding_signature` added alongside the promotion: a constant `sha2(capability, 256)`, so an
ongoing gap is one incident whose `detection_count` climbs and whose `resolved_at` auto-clears
the moment the capability's shift-context fields start populating again — needed now that this
feeds `ptof_obs_alert.ipynb`'s table-backed `INCIDENT_SOURCES` MERGE loop instead of a plain
WARN `check()`.

## ETL table staleness (added 2026-09-21, CRITICAL)
Per-table companion to the existing global `etl_pipeline_staleness` scalar check. Ships as
CRITICAL from day one (not staged as provisional/WARN like the 2026-09-18 MAD-based detectors)
because it reuses the already-proven simple-interval pattern (`minutes_since_last_run > N`)
rather than introducing a new unvalidated statistical model. Grouped directly off
`v_etl_bronze` — no `capability_registry`-style seed list needed, so coverage can't silently
drift if a table is added or removed upstream. Keyed on a constant `sha2(table_or_view, 256)`
(not table + window) so an ongoing stall is one incident whose `detection_count` climbs and
whose `resolved_at` auto-clears the moment the table resumes — same lifecycle as
`pipeline_heartbeat`/`etl_pipeline_staleness`, not the hourly-rollup pattern used by
`etl_run_slow`.

## ETL run slow (added 2026-09-18)
Introduced WARN-tier and provisional: `ptof_obs_alert.ipynb` computed and printed it to the job
log, never persisted to `obs_incidents` and never posted to Teams. This was deliberate: a
newly-introduced MAD-based threshold (see `threshold_basis`, originally `status='provisional'`)
that hadn't been validated against real incident history yet, so it ran in observe-only mode
until proven not to be noisy. Row-level anomalies roll up into an hourly bin requiring >=3
occurrences before it's written at all, so isolated blips never even reach the findings table.
Catches "runs completing, but slower than usual" — distinct from `etl_pipeline_health` (outright
failure) and `etl_pipeline_staleness` (no run at all).
**Grain changed 2026-09-21 (item #4, signed off):** rolled up from
`(table_or_view, task_name, window_start)` to `(task_name, window_start)` with
`affected_tables`/`affected_table_count` in the payload, because all tables under one
task_name move together — the old per-table grain produced 14-19 near-duplicate rows per
real event. **Promoted WARN -> CRITICAL 2026-09-21 (FP/FN bias review priority 2, signed
off):** 5 real, correlated, multi-table slowdown events were observed in ~1 week while this
was log-only and invisible to a human — a proven-real signal, not a newly-introduced
unvalidated one. Now persists to `obs_incidents`; per-incident Teams notification is
suppressed in favor of a daily digest (2026-09-21, FP/FN bias review step 3, signed off
"should be changed to a digest") — see `ptof_obs_alert.ipynb`.
Not a substitute for true per-call inference-latency detection (`latency_ms` on
`ptof_primary__ai_llm_audit_log`), which is blocked — that table has 0 rows in prod.

## ETL row count anomaly (added 2026-09-21, bronze-projection gap review)
Closes a blind spot shared by every other ETL detector above: a run can report
`status='success'`, land on schedule, and take a normal amount of time, while silently writing
zero rows — none of `etl_pipeline_health`/`etl_table_staleness`/`etl_run_slow` look at
`rows_written` at all. Live-data check across ~45,000 historical runs found the pattern is
cleanly bimodal, not continuous: 14 source tables have written >0 rows on 100% of their runs
(zero exceptions ever), and 5 tables (materialized-view refreshes + the validation gate) have
written exactly 0 rows on 100% of their runs (also zero exceptions) — no table has ever crossed
between the two regimes. The detector flags the first-ever crossing rather than an arbitrary
row-count threshold, since the historical data supports a binary regime, not a magnitude bound.
Shipped CRITICAL and digest-routed from day one (not staged WARN-first like the 2026-09-18 MAD
detectors) because the underlying baseline logic is a simple historical-regime check, not a new
unvalidated statistical model — and its fan-out shape (per-table, ETL-fleet-wide) is the same
shape `etl_table_staleness` had before its real 14/19-table event forced a digest conversion;
starting there avoids repeating that same lesson.

## Dropped detectors (prod migration 2026-09-10)
- `latency_anomalies` / `latency_anomaly_findings` — no `latency_ms` in prod
- `credential_fastfail_daily` — dev-specific model doesn't exist in prod
- `write_lag_daily` — depends on `latency_ms` for ingest-only computation
- `latency_failures` — depends on `success`, `error_class` (not in prod)
- `capability_health` / `capability_error_rate_alert` / `capability_error_rate_findings` — depends on `success`
- `prompt_size_drift` — no `user_prompt_chars` or `capability_latency_baseline` in prod

In [ ]:
%sql
-- capability_silence — CRITICAL liveness check (promoted from WARN 2026-09-21, detector value
-- audit follow-up, signed off): has each registered capability produced output within its
-- silence_grace_hours window? Joins capability_registry (active, non-null grace) against
-- v_llm_bronze (where output_type is aliased as capability, generated_at as called_at).
-- Prod capabilities: saa-display (2h), situational-awareness (2h), summary (36h).
-- sev2-insights has silence_grace_hours = NULL (irregular cadence) and is excluded by the WHERE
-- (covered instead by capability_silence_ceiling).
--
-- Promotion basis: full historical call-gap distribution for the 3 covered capabilities shows
-- 0 empirical false positives at any point (saa-display/situational-awareness: 4,815 gaps each,
-- max ever 0.27h vs 2h grace, 7.4x margin; summary: 96 gaps, max ever 12.04h vs 36h grace, 3.0x
-- margin) -- WARN-forever was hiding the exact single-capability-outage failure mode this
-- detector exists to catch, with a threshold that has real margin, not slack from being loose.
-- finding_signature added alongside the promotion: a constant sha2(capability, 256), same
-- pattern as capability_silence_ceiling/shift_context_missing -- an ongoing silence is one
-- incident whose detection_count climbs and whose resolved_at auto-clears the moment the
-- capability produces output again, rather than a fresh row every run. Needed now that this
-- feeds ptof_obs_alert.ipynb's table-backed INCIDENT_SOURCES MERGE loop instead of a plain
-- WARN check().
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.capability_silence AS
SELECT
    r.capability, r.expected_min_daily, r.silence_grace_hours, r.owner,
    count(b.id)      AS calls_last_7d,
    max(b.called_at) AS last_call_at,
    round((unix_timestamp(current_timestamp()) - unix_timestamp(max(b.called_at)))/3600.0, 1)
                     AS hours_since_last_call,
    sha2(r.capability, 256) AS finding_signature,
    current_timestamp()     AS detected_at
FROM mq_gmdf_dev.oil_obs.capability_registry r
LEFT JOIN mq_gmdf_dev.oil_obs.v_llm_bronze b
       ON b.capability = r.capability
      AND b.called_at >= current_timestamp() - INTERVAL 7 DAYS
WHERE r.active = true AND r.silence_grace_hours IS NOT NULL
GROUP BY 1, 2, 3, 4
HAVING max(b.called_at) IS NULL
    OR unix_timestamp(current_timestamp()) - unix_timestamp(max(b.called_at))
       > r.silence_grace_hours * 3600;

In [ ]:
%sql
-- capability_silence_ceiling (added 2026-09-21, item #2; promoted WARN -> CRITICAL and
-- tightened 168h -> 120h 2026-09-21, FP/FN bias review follow-up, signed off) -- backstop for
-- capabilities structurally EXCLUDED from capability_silence above because silence_grace_hours
-- IS NULL (irregular cadence). Confirmed gap (see HANDOFF.md "Detector value audit", #8):
-- sev2-insights has silence_grace_hours = NULL and had gone silent 64h+ with literally no
-- detection path at any length. This check does not replace the grace model -- it's a much
-- longer, fixed ceiling that only exists to catch "this capability has gone permanently dark,"
-- not routine irregular gaps (historical p95 0.2h, max 102.5h for sev2-insights).
--
-- silence_ceiling_hours is a separate column on capability_registry from silence_grace_hours
-- (NULL for every prod capability except sev2-insights, which is 120 -- tightened from 168 on
-- 2026-09-21: still 0/989 empirical false positives against sev2-insights's full gap history
-- (worst gap ever 102.5h), but cuts the false-negative exposure window vs. the original 168h).
-- CRITICAL as of 2026-09-21 -- promoted out of WARN because WARN-forever was itself judged a
-- false-negative risk (a real permanent-dark event would have gone unnotified indefinitely).
-- Caveat carried in threshold_basis: this has not yet fired on a real event, unlike etl_run_slow
-- before its own promotion -- revisit once it has some live-fire history.
--
-- finding_signature added 2026-09-21 alongside the CRITICAL promotion: a constant
-- sha2(capability, 256), same pattern as etl_table_staleness -- an ongoing silence is one
-- incident whose detection_count climbs and whose resolved_at auto-clears the moment the
-- capability produces output again, rather than a fresh row every run. Needed now that this
-- feeds ptof_obs_alert.ipynb's table-backed INCIDENT_SOURCES MERGE loop.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.capability_silence_ceiling AS
SELECT
    r.capability, r.silence_ceiling_hours, r.owner,
    max(b.called_at) AS last_call_at,
    round((unix_timestamp(current_timestamp()) - unix_timestamp(max(b.called_at)))/3600.0, 1)
                     AS hours_since_last_call,
    sha2(r.capability, 256) AS finding_signature,
    current_timestamp() AS detected_at
FROM mq_gmdf_dev.oil_obs.capability_registry r
LEFT JOIN mq_gmdf_dev.oil_obs.v_llm_bronze b
       ON b.capability = r.capability
WHERE r.active = true
  AND r.silence_grace_hours IS NULL
  AND r.silence_ceiling_hours IS NOT NULL
GROUP BY r.capability, r.silence_ceiling_hours, r.owner
HAVING max(b.called_at) IS NULL
    OR unix_timestamp(current_timestamp()) - unix_timestamp(max(b.called_at))
       > r.silence_ceiling_hours * 3600;

In [ ]:
%sql
-- shift_context_missing — data-quality check: detects output records where shift context
-- fields (shift_date, shift_type, batch_nbr) are blank or null. These fields enable per-shift
-- and per-batch slicing and AI-to-ISH correlation. Reads v_llm_bronze (where output_type is
-- aliased as capability, generated_at as called_at). Scoped to active capabilities via
-- capability_registry inner join.
--
-- Promoted WARN -> CRITICAL (2026-09-21, FP/FN bias review step 3, signed off): a standing
-- data-quality gap that silently degrades every downstream AI-to-ISH correlation join was
-- judged not worth leaving invisible to a human indefinitely, even though (like
-- capability_silence_ceiling before it) it hasn't been separately re-validated with a fresh
-- empirical FP sweep -- this mirrors that same promote-on-principle call, not a new
-- measurement. finding_signature added alongside the promotion: a constant sha2(capability,
-- 256), same pattern as capability_silence_ceiling/etl_table_staleness -- an ongoing gap is one
-- incident whose detection_count climbs and whose resolved_at auto-clears the moment the
-- capability's shift-context fields start populating again, rather than a fresh row every run.
-- Needed now that this feeds ptof_obs_alert.ipynb's table-backed INCIDENT_SOURCES MERGE loop.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.shift_context_missing AS
SELECT
    b.capability,
    count(*)                                                 AS total_calls,
    count_if(coalesce(b.shift_type, '') = '')                AS blank_shift_type,
    count_if(coalesce(cast(b.batch_nbr AS STRING), '') = '') AS blank_batch_nbr,
    count_if(b.shift_date IS NULL)                           AS null_shift_date,
    max(b.called_at)                                         AS last_seen,
    sha2(b.capability, 256)                                  AS finding_signature,
    current_timestamp()                                      AS detected_at
FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN mq_gmdf_dev.oil_obs.capability_registry r
  ON r.capability = b.capability AND r.active = true
WHERE b.called_at >= current_timestamp() - INTERVAL 7 DAYS
GROUP BY b.capability
HAVING count_if(coalesce(b.shift_type, '') = '') > 0
    OR count_if(coalesce(cast(b.batch_nbr AS STRING), '') = '') > 0
    OR count_if(b.shift_date IS NULL) > 0;

In [ ]:
%sql
-- etl_pipeline_health — CRITICAL detector: captures any ETL task failure in the last 24 hours.
-- The upstream ETL refreshes ~19 source tables every 10-15 min. When a task fails, the SAA
-- agent continues producing outputs using stale data — capability_silence and pipeline_heartbeat
-- won't fire because the agent is still generating, making this the only early warning.
-- Reads v_etl_bronze (pass-through view over mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit).
--
-- finding_signature changed 2026-09-21 (detector value audit, notify-routing review) from
-- sha2(table_or_view, run_id) to sha2(table_or_view) alone: keying on run_id meant a table
-- failing on repeated retries would open a NEW incident per retry instead of one standing
-- incident whose detection_count climbs -- the same bug class etl_table_staleness/
-- capability_silence/etc. are deliberately built to avoid. run_id was never in the
-- ptof_obs_alert.ipynb notify payload, so nothing is lost by dropping it from the key; the
-- payload still carries the latest run_timestamp/error_message/duration_seconds on each
-- re-detection. Historical check: only 1 failure row (1 table, 1 day) in the last 90 days, so
-- this had not yet caused a real duplicate-incident problem -- fixed proactively rather than
-- waiting for it to.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.etl_pipeline_health AS
SELECT
    run_id,
    run_timestamp,
    table_or_view,
    status,
    error_message,
    duration_seconds,
    sha2(table_or_view, 256) AS finding_signature,
    current_timestamp() AS detected_at
FROM mq_gmdf_dev.oil_obs.v_etl_bronze
WHERE status = 'failure'
  AND run_timestamp >= current_timestamp() - INTERVAL 24 HOURS;

In [ ]:
%sql
-- etl_table_staleness (added 2026-09-21, CRITICAL from day one) -- per-table companion to the
-- global etl_pipeline_staleness scalar check in ptof_obs_alert.ipynb. That global check takes
-- max(run_timestamp) across all 19 ETL source tables combined, which the 2026-09-19/20 weekend
-- incident showed can mask a partial stall: 14 of 19 tables went silent for ~25h while the
-- other 5 kept running on schedule, so the global max never went stale and nothing fired. This
-- check GROUPs BY table_or_view directly off v_etl_bronze (no capability_registry-style seed
-- list, so coverage can't drift if a table is added/removed upstream) and flags any single
-- table whose most recent run is more than 60 minutes old -- real fleet-wide cadence is
-- consistently ~12-16 min, so 60 min is a 4-6x margin above normal jitter, not a per-cycle
-- trigger.
--
-- Keyed on a constant sha2(table_or_view, 256) (not table + window), same lifecycle pattern as
-- pipeline_heartbeat/etl_pipeline_staleness: an ongoing stall is one incident whose
-- detection_count climbs in ptof_obs_alert.ipynb's MERGE loop, and whose resolved_at
-- auto-clears the moment the table resumes -- not a fresh row every run like the hourly-rollup
-- pattern used by write_lag_anomalies/etl_run_slow.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.etl_table_staleness AS
SELECT
    table_or_view,
    max(run_timestamp) AS last_run_at,
    round((unix_timestamp(current_timestamp()) - unix_timestamp(max(run_timestamp)))/60.0, 1)
                        AS minutes_since_last_run,
    sha2(table_or_view, 256) AS finding_signature,
    current_timestamp()      AS detected_at
FROM mq_gmdf_dev.oil_obs.v_etl_bronze
GROUP BY table_or_view
HAVING unix_timestamp(current_timestamp()) - unix_timestamp(max(run_timestamp)) > 60 * 60;

In [ ]:
%sql
-- etl_run_slow (added 2026-09-18, WARN, provisional; promoted to CRITICAL 2026-09-21 -- see
-- threshold_basis) -- flags v_etl_bronze runs whose duration_seconds exceeds that
-- (table_or_view, task_name)'s MAD-based upper_bound_s from etl_duration_baseline. Distinct
-- from etl_pipeline_health (which catches outright failures) and etl_pipeline_staleness (which
-- catches "no run completed at all") -- this catches "runs are completing, but taking longer
-- than their own history," an early warning that can precede an actual failure or staleness
-- incident.
--
-- Grain (changed 2026-09-21, item #4, signed off): rolled up to (task_name, window_start),
-- NOT (table_or_view, task_name, window_start). The audit found all tables under one task_name
-- always move together -- the 4 real slowdown events in 7 days each hit 14-19 of 19 tables
-- simultaneously, so the old per-table grain produced 14-19 near-duplicate rows for a single
-- real event. affected_tables/affected_table_count preserve which tables were involved without
-- fragmenting one event into many rows. Row-level slow runs are still logged per-table, but the
-- >=3-in-an-hour rollup that decides whether anything gets written now groups by task_name.
--
-- Promoted WARN -> CRITICAL (2026-09-21, FP/FN bias review priority 2, signed off): 5 real,
-- correlated, multi-table slowdown events were observed in ~1 week while this was log-only and
-- invisible to a human -- a proven-real signal. ptof_obs_alert.ipynb now persists this to
-- obs_incidents and posts it to Teams via the standard CRITICAL lifecycle.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.etl_run_slow AS
WITH flagged AS (
    SELECT
        e.table_or_view, e.task_name, e.duration_seconds, e.run_timestamp,
        window(e.run_timestamp, '60 minutes').start AS window_start,
        base.upper_bound_s
    FROM mq_gmdf_dev.oil_obs.v_etl_bronze e
    JOIN mq_gmdf_dev.oil_obs.etl_duration_baseline base
      ON base.table_or_view = e.table_or_view AND base.task_name = e.task_name
    WHERE e.run_timestamp >= current_timestamp() - INTERVAL 24 HOURS
      AND e.status = 'success'
      AND e.duration_seconds > base.upper_bound_s
)
SELECT
    task_name, window_start,
    count(*)                                       AS anomalous_count,
    count(distinct table_or_view)                  AS affected_table_count,
    concat_ws(', ', collect_set(table_or_view))     AS affected_tables,
    max(duration_seconds)                          AS max_duration_s,
    max(upper_bound_s)                             AS upper_bound_s,
    sha2(concat_ws('|', task_name, cast(window_start AS STRING)), 256)
                                                     AS finding_signature,
    current_timestamp()                            AS detected_at
FROM flagged
GROUP BY task_name, window_start
HAVING count(*) >= 3;

In [ ]:
%sql
-- etl_row_count_anomaly — added 2026-09-21 (bronze-projection gap review, live-data confirmed).
-- Closes a blind spot etl_pipeline_failure/etl_table_staleness/etl_run_slow all share: a run can
-- report status='success', land on schedule, and take a normal amount of time, while silently
-- writing zero rows -- a partial/no-op load none of the other 3 detectors can see, because none
-- of them look at rows_written.
--
-- Baseline is computed inline from full history rather than a separate nightly baseline table
-- (unlike etl_run_slow/etl_duration_baseline) because the pattern is dead simple and bimodal,
-- not a continuous distribution needing MAD: across ~45,000 historical runs, 14 source tables
-- have written >0 rows on literally every run ever (0 exceptions), and 5 tables (materialized-
-- view refreshes + the validation gate) have written exactly 0 rows on every run ever (also 0
-- exceptions) -- that's just how those jobs work, not a problem. No table has ever crossed
-- between the two regimes. >=50-run floor (pct_zero >= 0.99 / <= 0.01) keeps a thinly-run table
-- from getting a false regime off too little history; anything in between is 'irregular' and
-- deliberately excluded -- there's no clean baseline to violate.
--
-- finding_signature is a constant sha2(table_or_view, 256), same standing-condition lifecycle
-- as etl_table_staleness. Notify routing: digest, not per-incident, from day one (see
-- ptof_obs_alert.ipynb) -- etl_table_staleness's real 14/19-table simultaneous event is the
-- closest analog for this detector's fan-out shape, so it starts where that one ended up
-- rather than risking the same flood and needing a second conversion later.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.etl_row_count_anomaly AS
WITH baseline AS (
    SELECT table_or_view,
           count(*) AS n,
           avg(CASE WHEN rows_written = 0 THEN 1.0 ELSE 0.0 END) AS pct_zero
    FROM mq_gmdf_dev.oil_obs.v_etl_bronze
    WHERE status = 'success'
    GROUP BY table_or_view
    HAVING count(*) >= 50
),
regime AS (
    SELECT table_or_view,
           CASE WHEN pct_zero >= 0.99 THEN 'expect_zero'
                WHEN pct_zero <= 0.01 THEN 'expect_nonzero'
                ELSE 'irregular' END AS regime
    FROM baseline
),
recent AS (
    SELECT table_or_view, run_timestamp, rows_written
    FROM mq_gmdf_dev.oil_obs.v_etl_bronze
    WHERE status = 'success'
      AND run_timestamp >= current_timestamp() - INTERVAL 24 HOURS
)
SELECT
    r.table_or_view,
    r.rows_written,
    g.regime,
    r.run_timestamp,
    sha2(r.table_or_view, 256) AS finding_signature,
    current_timestamp() AS detected_at
FROM recent r
JOIN regime g ON g.table_or_view = r.table_or_view
WHERE (g.regime = 'expect_nonzero' AND r.rows_written = 0)
   OR (g.regime = 'expect_zero' AND r.rows_written > 0);